# 벌통 상태 분류 - 실험B 적용

원본 녹음 단위로 층화 분할한 뒤, GAP CNN을 먼저 학습하고 고정합니다. 음향 특징에 현재 환경값과 시간대 특징을 결합한 벡터로 MLP를 학습합니다.

결과는 `artifacts/main/<실행 ID>/`에 저장합니다.

## 1. 실행 설정과 Autotune

기본 설정은 batch 64, patience 5, epoch 50

Autotune을 True로 바꿔야 Autotune 켜집니다.

새 학습을 시작하려면 EVALUATE_RUN_DIR를 None으로 두세요.

기존 모델로 평가만 다시 하려면 `EVALUATE_RUN_DIR`에 실행 폴더 경로를 지정하고 위에서부터 실행합니다. 저장된 모델·전처리·WAV 순서를 사용합니다.

In [ ]:
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from datetime import datetime
from hashlib import sha256
from pathlib import Path
from time import perf_counter
from uuid import uuid4
import json
import math
import re
import tempfile

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
from IPython.display import Image, display
from keras import callbacks, layers, models, optimizers
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

OPTIONS = globals().get('RUN_OPTIONS', {})
ROOT = Path(OPTIONS.get('root', Path.cwd())).resolve()
SEED = 42
BATCH_SIZE = 64
MAX_EPOCHS = int(OPTIONS.get('epochs', 50))
PATIENCE = 5
AUTOTUNE = bool(OPTIONS.get('autotune', True))  # True: 각 단계에서 검증 손실로 조합 선택
LEARNING_RATES = (0.001, 0.0005)
DROPOUT_RATES = (0.3, 0.2)
DEFAULT_PARAMS = {'learning_rate': LEARNING_RATES[0], 'dropout': DROPOUT_RATES[0]}
SILENCE_RMS = 1e-5
SPLITS = ('train', 'val', 'test')
CLASSES = np.arange(4)
LABEL_MAP = {label: f'queen status {label}' for label in range(4)}
SCORE_COLUMNS = [f'score_{label}' for label in CLASSES]
ENV_COLUMNS = ['hive temp', 'hive humidity', 'hive pressure', 'weather temp', 'weather pressure']

run_id = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid4().hex[:8]
# 기존 실행을 평가만 할 때 실행 폴더를 지정합니다.
EVALUATE_RUN_DIR = OPTIONS.get('evaluate_run_dir', None)
if EVALUATE_RUN_DIR is None:
    OUTPUT = Path(OPTIONS.get('output_root', ROOT / 'artifacts/main' / run_id)).resolve()
    OUTPUT.mkdir(parents=True, exist_ok=False)
    saved_settings = None
else:
    OUTPUT = Path(EVALUATE_RUN_DIR).resolve()
    settings_path = OUTPUT / 'cnn_mlp/preprocessing.json'
    saved_settings = json.loads(settings_path.read_text(encoding='utf-8'))
    if saved_settings.get('experiment') != 'B':
        raise ValueError(f'실험 B의 모델 폴더를 지정하세요. {OUTPUT}')
    if 'autotune' in saved_settings:
        AUTOTUNE = saved_settings['autotune']
    else:
        AUTOTUNE = len(pd.read_csv(OUTPUT / 'cnn_mlp/tuning_trials.csv')) > 1
    SILENCE_RMS = saved_settings['quality']['silence_rms']
print(f'결과 폴더  {OUTPUT}')

def save_json(path, data):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

## 2. WAV 연결과 원본 단위 배정

WAV를 CSV의 원본 관측에 연결합니다. 고정 배정 CSV가 있으면 원본 키, 라벨, 벌통, 관측 시각, WAV 수를 대조합니다. CSV가 없으면 동일한 원본 목록에서 seed 42로 층화 분할을 재현합니다.

In [ ]:
def hive_key(value):
    if value is None or pd.isna(value) or str(value).strip().lower() in ('', 'nan', 'none'):
        raise ValueError('Missing hive identifier')
    text = str(value).strip()
    try:
        number = float(text)
    except ValueError:
        return text
    if not np.isfinite(number) or number <= 0 or not number.is_integer():
        raise ValueError('Invalid hive identifier')
    return str(int(number))


def timestamp(value):
    value = pd.Timestamp(value)
    if pd.isna(value) or value.tzinfo is not None:
        raise ValueError('A valid local timestamp without timezone is required')
    return value


def prepare_metadata(raw):
    required = {'file name', 'hive number', 'date', 'queen status', *ENV_COLUMNS}
    if required - set(raw):
        raise ValueError(f'Missing CSV columns: {required - set(raw)}')
    rows, exclusions = [], []
    for index, row in raw.iterrows():
        try:
            name = row['file name']
            if pd.isna(name) or not str(name).strip():
                raise ValueError('missing source_id')
            hive = float(row['hive number'])
            if not np.isfinite(hive) or hive <= 0 or not hive.is_integer():
                raise ValueError('invalid hive identifier')
            observed = timestamp(row['date'])
            record = {'csv_row': int(index), 'source_id': Path(str(name)).stem,
                      'hive_id': hive_key(hive), 'observed_at': observed,
                      'session_day': observed.strftime('%Y-%m-%d'),
                      'label': pd.to_numeric(row['queen status'], errors='coerce')}
            record.update({column: pd.to_numeric(row[column], errors='coerce')
                           for column in ENV_COLUMNS})
            rows.append(record)
        except (ValueError, TypeError, OverflowError) as error:
            exclusions.append({'csv_row': index, 'source_id': row.get('file name'),
                               'reason': str(error)})
    result = pd.DataFrame(rows)
    if result.empty:
        raise ValueError('No valid original environment observations')
    # An ambiguous source filename cannot be linked to a WAV safely.
    for source, group in result.groupby('source_id', sort=False):
        if len(group.drop(columns='csv_row').drop_duplicates()) != 1:
            raise ValueError(f'Conflicting CSV source: {source}')
    return result.drop_duplicates('source_id').reset_index(drop=True), pd.DataFrame(
        exclusions, columns=['csv_row', 'source_id', 'reason'])


def scan_project(root, output_dir):
    root, output_dir = Path(root), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    raw = pd.read_csv(root / 'dataset/all_data_updated.csv')
    metadata, invalid = prepare_metadata(raw)
    index = metadata.set_index('source_id')
    records, excluded = [], []
    wav_paths = sorted((root / 'dataset').rglob('*.wav'))
    for path in wav_paths:
        relative = path.relative_to(root).as_posix()
        matched = re.fullmatch(r'(.+)__segment(\d+)', path.stem)
        source = matched.group(1) if matched else path.stem
        segment = int(matched.group(2)) if matched else 0
        try:
            if source not in index.index:
                raise ValueError('CSV match missing or invalid observation identifier/time')
            row = index.loc[source]
            if pd.isna(row['label']) or row['label'] not in LABEL_MAP:
                raise ValueError('Invalid queen status label')
            info = sf.info(path)
            if not 0 < info.duration <= 60.0:
                raise ValueError('WAV must contain >0 and <=60 seconds')
            record = row.to_dict()
            record.update({'source_id': source, 'file_path': relative,
                           'label': int(row['label']), 'segment': segment,
                           'recorded_at': row['observed_at'] + pd.Timedelta(seconds=60 * segment),
                           'duration': info.duration, 'sample_rate': info.samplerate,
                           'channels': info.channels, 'subtype': info.subtype})
            records.append(record)
        except (OSError, RuntimeError, ValueError) as error:
            excluded.append({'file_path': relative, 'reason': str(error)})
    pd.DataFrame(excluded, columns=['file_path', 'reason']).to_csv(
        output_dir / 'matching_exclusions.csv', index=False)
    invalid.to_csv(output_dir / 'environment_exclusions.csv', index=False)
    if not records:
        raise ValueError('No usable WAVs in dataset')
    manifest = pd.DataFrame(records)
    if not manifest['file_path'].is_unique:
        raise ValueError('Duplicate WAV paths')
    return manifest, metadata

def assign_sources(manifest):
    source_fields = ['hive_id', 'label', 'observed_at', 'session_day']
    sources = manifest[['source_id', *source_fields]].drop_duplicates().set_index('source_id').sort_index()
    if not sources.index.is_unique:
        raise ValueError('한 원본에 서로 다른 라벨·벌통·관측 시각이 연결되어 있습니다.')
    sources['wav_count'] = manifest.groupby('source_id').size()
    fields = [*source_fields, 'wav_count']
    assignment_path = ROOT / 'docs/experiment_splits/b_source_assignments.csv'
    if assignment_path.is_file():
        fixed = pd.read_csv(assignment_path, dtype={'source_id': str, 'hive_id': str}).set_index('source_id')
        fixed['observed_at'] = pd.to_datetime(fixed['observed_at'])
        if not fixed.index.is_unique or set(fixed['split']) != set(SPLITS):
            raise ValueError('배정 CSV의 원본 키 또는 split을 확인하세요.')
        keys = sources.index.union(fixed.index)
        differences = sources.reindex(keys)[fields].compare(
            fixed.reindex(keys)[fields], result_names=('local', 'assignment'))
        if not differences.empty or set(sources.index) != set(fixed.index):
            differences.to_csv(OUTPUT / 'assignment_mismatches.csv')
            raise ValueError(f'로컬 원본과 고정 배정이 다릅니다. {OUTPUT / "assignment_mismatches.csv"}')
        sources['split'] = fixed['split']
        origin = 'fixed_csv'
    else:
        # 동일한 원본 목록에서 B의 70/15/15 층화 배정을 재현합니다.
        train, held = train_test_split(sources, test_size=0.30, stratify=sources['label'], random_state=SEED)
        val, test = train_test_split(held, test_size=0.50, stratify=held['label'], random_state=SEED)
        for split, part in zip(SPLITS, (train, val, test)):
            sources.loc[part.index, 'split'] = split
        origin = 'stratified_seed_42'
    sources['observed_hour'] = sources['observed_at'].dt.floor('h')
    sources.to_csv(OUTPUT / 'source_assignments.csv')
    save_json(OUTPUT / 'split_config.json', {'origin': origin, 'seed': SEED, 'unit': 'source_id'})
    return manifest.assign(split=manifest['source_id'].map(sources['split']))

if saved_settings is None:
    local_wavs, _ = scan_project(ROOT, OUTPUT)
    assigned_wavs = assign_sources(local_wavs)
else:
    assigned_wavs = pd.read_csv(
        OUTPUT / 'manifest.csv', dtype={'source_id': str, 'hive_id': str}, float_precision='round_trip',
        parse_dates=['observed_at', 'recorded_at'],
    )

### 데이터 탐색 (EDA)

아래 그래프의 클래스 번호는 CSV의 `queen status`와 같습니다.

| 라벨 | 의미 |
| --- | --- |
| 0 | 기존 여왕벌 존재 |
| 1 | 여왕벌 없음 |
| 2 | 새 여왕벌 거부됨 |
| 3 | 새 여왕벌 수용됨 |

바로 위에서 불러온 목록을 바탕으로 분할별 원본 녹음과 WAV 수를 비교합니다. 센서값은 원본 관측당 한 번만 집계하고, 외기 기압 0은 결측으로 처리합니다.

In [ ]:
eda_sources = assigned_wavs.drop_duplicates('source_id')
eda_count_figure, eda_count_axes = plt.subplots(1, 2, figsize=(11, 3.5), layout='constrained')
for ax, rows, title in zip(eda_count_axes, (eda_sources, assigned_wavs),
                           ('Original recordings by split', 'WAV files by split')):
    counts = pd.crosstab(rows['label'], rows['split']).reindex(
        index=CLASSES, columns=SPLITS, fill_value=0)
    counts.plot.bar(ax=ax, rot=0, color=['#22639D', '#B86B26', '#378367'])
    ax.set(title=title, xlabel='Queen status', ylabel='Count')

eda_environment = eda_sources[ENV_COLUMNS].replace([np.inf, -np.inf], np.nan)
eda_environment['weather pressure'] = eda_environment['weather pressure'].replace(0, np.nan)
eda_environment_figure, eda_environment_axes = plt.subplots(2, 3, figsize=(11, 6), layout='constrained')
eda_environment_figure.suptitle('Sensor readings by queen status')
for ax, column in zip(eda_environment_axes.flat, ENV_COLUMNS):
    readings = [eda_environment.loc[eda_sources['label'].eq(label), column].dropna().to_numpy()
                for label in CLASSES]
    ax.boxplot(readings)
    ax.set_xticks(range(1, len(CLASSES) + 1), CLASSES)
    ax.set(title=column, xlabel='Queen status')
eda_environment_axes[-1, -1].set_visible(False)

plt.show()
plt.close(eda_count_figure)
plt.close(eda_environment_figure)

## 3. 60초 WAV 로그멜 변환

모노 16kHz 음성에 대해 기존과 같은 로그멜을 계산합니다. WAV 하나가 CNN 입력 `(64, 1876, 1)` 하나에 대응합니다.

In [ ]:
@dataclass(frozen=True)
class AudioConfig:
    sample_rate: int = 16000
    segment_seconds: float = 60.0  # WAV 하나 전체를 모델 입력 하나로 사용한다.
    n_mels: int = 64
    n_fft: int = 1024
    hop_length: int = 512
    fmin: float = 0.0
    fmax: float = 8000.0
    top_db: float = 80.0

    def __post_init__(self):
        if min(self.sample_rate, self.segment_seconds, self.n_mels,
               self.n_fft, self.hop_length) <= 0:
            raise ValueError("전처리 크기와 시간은 양수여야 합니다.")
        if not 0 <= self.fmin < self.fmax <= self.sample_rate / 2:
            raise ValueError("멜 주파수 범위를 확인하세요.")

    @property
    def samples(self):
        return round(self.sample_rate * self.segment_seconds)

    @property
    def shape(self):
        return (self.n_mels, 1 + self.samples // self.hop_length, 1)


def read_audio(source, config, offset=0.0, duration=None):
    """WAV의 지정 구간을 mono/고정 샘플링 주파수로 읽는다. 패딩은 여기서 하지 않는다."""
    if offset < 0 or (duration is not None and duration <= 0):
        raise ValueError("시작 시각과 구간 길이를 확인하세요.")
    if hasattr(source, "seek"):
        source.seek(0)
    with sf.SoundFile(source) as audio:
        original_sr = audio.samplerate
        waveform = audio.read(dtype="float32", always_2d=True)
    if not len(waveform):
        raise ValueError("읽을 수 있는 음성 구간이 없습니다.")
    if not np.isfinite(waveform).all():
        raise ValueError("음성에 NaN 또는 무한대 값이 있습니다.")
    waveform = waveform.mean(axis=1)
    if original_sr != config.sample_rate:
        waveform = librosa.resample(
            waveform, orig_sr=original_sr, target_sr=config.sample_rate,
            res_type="soxr_hq",
        )
    # 먼저 동일한 전체 녹음을 리샘플링한 뒤 자른다. 학습/앱 구간 경계도 같아진다.
    start = round(offset * config.sample_rate)
    end = None if duration is None else start + round(duration * config.sample_rate)
    waveform = waveform[start:end]
    if not len(waveform):
        raise ValueError("읽을 수 있는 음성 구간이 없습니다.")
    return np.asarray(waveform, dtype="float32")


def log_mel(waveform, config):
    """녹음 전체를 (mel, time, channel) 로그 멜 배열로 변환한다."""
    waveform = np.asarray(waveform, dtype="float32")
    if waveform.ndim != 1 or not len(waveform) or not np.isfinite(waveform).all():
        raise ValueError("비어 있지 않은 유한한 mono 음성이 필요합니다.")
    if len(waveform) > config.samples:
        raise ValueError("녹음 길이가 모델 입력 길이와 맞지 않습니다.")
    waveform = librosa.util.fix_length(waveform, size=config.samples)
    mel = librosa.feature.melspectrogram(
        y=waveform, sr=config.sample_rate, n_fft=config.n_fft,
        hop_length=config.hop_length, n_mels=config.n_mels,
        fmin=config.fmin, fmax=config.fmax, power=2.0,
        center=True, pad_mode="constant",
    )
    return librosa.power_to_db(mel, ref=1.0, top_db=config.top_db)[..., None].astype("float32")


def audio_quality(waveform):
    """RMS·최대 진폭·±1 이상 샘플 비율을 계산한다."""
    return {
        "rms": float(np.sqrt(np.mean(np.square(waveform, dtype="float64")))),
        "peak": float(np.max(np.abs(waveform))),
        "over_full_scale_ratio": float(np.mean(np.abs(waveform) >= 1.0)),
    }


CONFIG = AudioConfig(**saved_settings['audio']) if saved_settings else AudioConfig()

## 4. 품질 검사와 특징 캐시

음성 전처리 설정, 파일 키, 행 순서, 파일 정보가 맞는 캐시를 읽기 전용으로 사용합니다. 캐시에 없는 WAV는 `artifacts/main/feature_cache/`에 계산해 저장하고 다음 실행에서 재사용합니다.

In [ ]:
class IndexedAudio:
    """파일 키를 로그멜 배열의 행에 연결하고 필요한 배치만 읽습니다."""
    def __init__(self, arrays, locations, file_paths):
        self.arrays = arrays
        self.rows = np.array([locations[file] for file in file_paths], dtype=int)
        self.shape = (len(self.rows), *CONFIG.shape)

    def __len__(self):
        return self.shape[0]

    def __getitem__(self, selection):
        rows = self.rows[selection]
        if rows.ndim == 1:
            shard, index = rows
            return self.arrays[shard][index]
        batch = np.empty((len(rows), *CONFIG.shape), dtype=np.float32)
        for shard in np.unique(rows[:, 0]):
            mask = rows[:, 0] == shard
            batch[mask] = self.arrays[shard][rows[mask, 1]]
        return batch


def wav_stamp(file):
    stat = (ROOT / file).stat()
    return f'{stat.st_size}:{stat.st_mtime_ns}'


def cached_extraction(file_paths, stamps, settings):
    files = sorted(file_paths)
    identity = json.dumps([settings, [(file, stamps[file]) for file in files]], sort_keys=True)
    key = sha256(identity.encode('utf-8')).hexdigest()
    root = ROOT / 'artifacts/main/feature_cache'
    root.mkdir(parents=True, exist_ok=True)
    shape = (len(files), *CONFIG.shape)

    def read_entry(directory):
        try:
            rows = pd.read_csv(directory / 'quality.csv').fillna({'reason': ''})
            if (rows['file_path'].tolist() != files or rows['wav_stamp'].tolist() != [stamps[f] for f in files]
                    or 'reason' not in rows):
                return None
            array = np.load(directory / 'features.npy', mmap_mode='r', allow_pickle=False)
            if array.shape == shape and array.dtype == np.float32:
                return array, rows.to_dict('records')
        except (OSError, ValueError, KeyError):
            return None

    for entry in sorted(root.glob(key + '*')):
        cached = read_entry(entry)
        if cached is not None:
            return cached

    # Publish the array and its row map together after a successful write.
    with tempfile.TemporaryDirectory(dir=root, prefix='building_') as temporary:
        pending = Path(temporary) / 'entry'
        pending.mkdir()
        features = np.lib.format.open_memmap(pending / 'features.npy', mode='w+', dtype='float32', shape=shape)
        rows = []
        for index, file in enumerate(files):
            row = {'file_path': file, 'wav_stamp': stamps[file], 'reason': ''}
            try:
                waveform = read_audio(ROOT / file, CONFIG)
                row.update(audio_quality(waveform))
                if row['rms'] < SILENCE_RMS:
                    raise ValueError('near-silent recording')
                features[index] = log_mel(waveform, CONFIG)
            except (OSError, RuntimeError, ValueError) as error:
                row['reason'] = str(error)
            rows.append(row)
            if (index + 1) % 200 == 0:
                print(f'로그멜 계산 {index + 1}/{len(files)}', flush=True)
        features.flush()
        del features
        pd.DataFrame(rows).to_csv(pending / 'quality.csv', index=False)
        destination = root / key
        try:
            pending.rename(destination)
        except OSError:
            if not destination.is_dir():
                raise
            cached = read_entry(destination)
            if cached is not None:
                return cached
            destination = root / f'{key}_{uuid4().hex}'
            pending.rename(destination)
    return np.load(destination / 'features.npy', mmap_mode='r', allow_pickle=False), rows


def load_audio_features(file_paths, expected_stamps=None):
    settings = {'audio': asdict(CONFIG), 'silence_rms': SILENCE_RMS, 'manual_exclusions': {}}
    stamps = {file: wav_stamp(file) for file in file_paths}
    if expected_stamps is not None and any(expected_stamps.get(file) != stamp for file, stamp in stamps.items()):
        raise ValueError('저장된 실행 이후 WAV가 바뀌었습니다. 원래 WAV로 평가하세요.')
    cache = ROOT / 'artifacts/recording_60s'
    arrays, locations, quality_rows = [], {}, {}
    try:
        original = pd.read_csv(cache / 'manifest.csv', usecols=['split', 'file_path'])
    except (OSError, ValueError):
        original = pd.DataFrame(columns=['split', 'file_path'])
    if not original.empty:
        for split in SPLITS:
            try:
                saved = json.loads((cache / 'features' / f'{split}_config.json').read_text(encoding='utf-8'))
                quality = pd.read_csv(cache / f'quality_{split}.csv').fillna({'reason': ''})
                order = original.loc[original['split'].eq(split), 'file_path'].tolist()
                if (saved != settings or not {'file_path', 'reason', 'wav_stamp'}.issubset(quality)
                        or quality['file_path'].tolist() != order or not quality['file_path'].is_unique):
                    continue
                array = np.load(cache / 'features' / f'{split}.npy', mmap_mode='r', allow_pickle=False)
                if array.shape != (len(quality), *CONFIG.shape) or array.dtype != np.float32:
                    continue
            except (OSError, ValueError):
                continue
            shard = len(arrays)
            arrays.append(array)
            index = 0
            for row in quality.to_dict('records'):
                file = row['file_path']
                valid = not row['reason']
                if stamps.get(file) == row['wav_stamp']:
                    quality_rows[file] = {**row, 'feature_origin': f'cache/{split}'}
                    if valid:
                        locations[file] = (shard, index)
                index += valid
    missing = [file for file in file_paths if file not in quality_rows]
    if missing:
        shard = len(arrays)
        array, rows = cached_extraction(missing, stamps, settings)
        arrays.append(array)
        for index, row in enumerate(rows):
            file = row['file_path']
            quality_rows[file] = {**row, 'feature_origin': 'persistent_cache'}
            if not row['reason']:
                locations[file] = (shard, index)
    quality = pd.DataFrame(quality_rows.values()).drop(columns=['split'], errors='ignore')
    if expected_stamps is None:
        quality.to_csv(OUTPUT / 'audio_quality.csv', index=False)
    print('로그멜 특징', quality['feature_origin'].value_counts().to_dict())
    return arrays, locations

## 5. 분할 안에서 WAV 순서 섞기

품질 검사 후 각 세트 안에서 WAV 행을 섞습니다. 파일 경로, 라벨, 특징은 같은 행을 따라갑니다. 클래스 분포와 B의 학습·테스트 시간 인접도를 저장합니다.

In [ ]:
expected_stamps = None
if saved_settings is not None:
    quality = pd.read_csv(OUTPUT / 'audio_quality.csv', usecols=['file_path', 'wav_stamp'])
    expected_stamps = quality.set_index('file_path')['wav_stamp'].to_dict()
arrays, locations = load_audio_features(assigned_wavs['file_path'].tolist(), expected_stamps)
usable = assigned_wavs[assigned_wavs['file_path'].isin(locations)].copy()
if saved_settings is not None and len(usable) != len(assigned_wavs):
    raise ValueError('저장된 실행의 WAV 중 특징을 만들 수 없는 파일이 있습니다.')
if saved_settings is None:
    sources = usable.drop_duplicates('source_id')
    train_sources = sources[sources['split'].eq('train')]
    test_sources = sources[sources['split'].eq('test')]
    neighbors = test_sources[['source_id', 'hive_id', 'session_day', 'observed_at']].merge(
        train_sources[['hive_id', 'session_day', 'observed_at']], on='hive_id', suffixes=('_test', '_train'))
    same_day = neighbors['session_day_test'].eq(neighbors['session_day_train'])
    within_hour = (neighbors['observed_at_test'] - neighbors['observed_at_train']).abs().le(pd.Timedelta(hours=1))
    save_json(OUTPUT / 'split_diagnostics.json', {
        'test_sources': len(test_sources), 'same_hive_day': int(neighbors.loc[same_day, 'source_id'].nunique()),
        'within_60_minutes': int(neighbors.loc[within_hour, 'source_id'].nunique()),
        'source_overlap': len(set(train_sources['source_id']) & set(test_sources['source_id'])),
    })

prepared = {}
for split in SPLITS:
    rows = usable[usable['split'].eq(split)]
    if saved_settings is None:
        rows = rows.sort_values('file_path')
        if OPTIONS.get('smoke_per_class'):
            rows = rows.groupby('label', group_keys=False).head(int(OPTIONS['smoke_per_class']))
        order = np.random.default_rng(SEED).permutation(len(rows))
        rows = rows.iloc[order]
    if set(rows['label']) != set(CLASSES):
        raise ValueError(f'{split} 분할의 품질 검사 후 네 클래스가 모두 필요합니다.')
    rows = rows.assign(offset_seconds=0.0).reset_index(drop=True)
    prepared[split] = {
        'rows': rows, 'audio': IndexedAudio(arrays, locations, rows['file_path']),
        'labels': rows['label'].to_numpy(dtype='int64'),
    }

manifest = pd.concat([part['rows'] for part in prepared.values()], ignore_index=True)
observations = manifest.drop_duplicates('source_id').sort_values(['hive_id', 'observed_at', 'source_id'])
if saved_settings is None:
    manifest.to_csv(OUTPUT / 'manifest.csv', index=False)
    observations.to_csv(OUTPUT / 'environment_observations.csv', index=False)
distribution = pd.concat([
    pd.crosstab(manifest['split'], manifest['label']),
    pd.crosstab(observations['split'], observations['label']),
], axis=1, keys=['WAV', '원본']).reindex(SPLITS)
if saved_settings is None:
    distribution.to_csv(OUTPUT / 'split_distribution.csv')
display(distribution)

## 6. 학습 자료로 전처리 통계 계산

학습 원본의 환경값으로 중앙값·평균·표준편차를 구합니다. 학습 WAV로 음성 정규화와 클래스 가중치를 계산합니다. 환경 입력은 현재 관측값 5개와 4시간 단위 시간대 원핫 6개입니다.

In [ ]:
def numeric_values(frame):
    values = frame[ENV_COLUMNS].apply(pd.to_numeric, errors='coerce').to_numpy(dtype='float64')
    values[~np.isfinite(values)] = np.nan
    values[values[:, 4] == 0, 4] = np.nan
    return values


def fit_environment(observations):
    train = observations[observations['split'].eq('train')].drop_duplicates('source_id')
    if train.empty:
        raise ValueError('학습 원본 환경 관측이 필요합니다.')
    values = numeric_values(train)
    median = np.array([np.nanmedian(col) if np.isfinite(col).any() else 0.0 for col in values.T])
    filled = np.where(np.isnan(values), median, values)
    std = filled.std(axis=0)
    return {'columns': ENV_COLUMNS, 'median': median.tolist(), 'mean': filled.mean(axis=0).tolist(),
            'std': np.where(std > 1e-8, std, 1.0).tolist(), 'training_observations': len(values)}


def current_environment(rows, settings):
    values = numeric_values(rows)
    filled = np.where(np.isnan(values), settings['median'], values)
    scaled = (filled - settings['mean']) / settings['std']
    time_bin = rows['recorded_at'].dt.hour.to_numpy() // 4
    return np.concatenate([scaled, np.eye(6)[time_bin]], axis=1).astype('float32')


def audio_statistics(audio):
    total = squares = count = 0
    for start in range(0, len(audio), BATCH_SIZE):
        batch = audio[start:start + BATCH_SIZE].astype('float64')
        total += batch.sum()
        squares += np.square(batch).sum()
        count += batch.size
    mean = float(total / count)
    return {'mean': mean, 'variance': max(float(squares / count - mean ** 2), 1e-6)}


def audio_dataset(part, shuffle=False):
    rng = np.random.default_rng(SEED)

    def batches():
        order = rng.permutation(len(part['labels'])) if shuffle else np.arange(len(part['labels']))
        for start in range(0, len(order), BATCH_SIZE):
            indices = order[start:start + BATCH_SIZE]
            yield part['audio'][indices], part['labels'][indices]

    signature = (tf.TensorSpec((None, *CONFIG.shape), tf.float32), tf.TensorSpec((None,), tf.int64))
    dataset = tf.data.Dataset.from_generator(batches, output_signature=signature)
    dataset = dataset.apply(tf.data.experimental.assert_cardinality(math.ceil(len(part['labels']) / BATCH_SIZE)))
    options = tf.data.Options()
    options.threading.private_threadpool_size = 2
    return dataset.with_options(options).prefetch(1)


if saved_settings is None:
    environment = fit_environment(observations)
    normalization = audio_statistics(prepared['train']['audio'])
    weights = compute_class_weight('balanced', classes=CLASSES, y=prepared['train']['labels'])
    class_weights = dict(enumerate(map(float, weights)))
else:
    environment = saved_settings['environment']
for part in prepared.values():
    part['environment'] = current_environment(part['rows'], environment)

## 7. GAP CNN과 MLP 정의

CNN은 60초 WAV에서 64차원 특징을 추출합니다. MLP는 음향 특징 64개와 환경·시간대 특징 11개를 합친 75차원 벡터를 받습니다.

In [ ]:
def make_cnn(learning_rate, dropout):
    cnn = models.Sequential(name='audio_cnn')
    cnn.add(layers.Input(shape=CONFIG.shape, name='audio_input'))
    cnn.add(layers.Normalization(axis=None, **normalization, name='audio_normalization'))
    for filters in (16, 32, 64):
        cnn.add(layers.Conv2D(filters, (3, 3), padding='same', activation='relu', name=f'conv_{filters}'))
        cnn.add(layers.MaxPooling2D((2, 2), name=f'pool_{filters}'))
    cnn.add(layers.GlobalAveragePooling2D(name='audio_global_average'))
    cnn.add(layers.Dense(64, activation='relu', name='audio_features'))
    cnn.add(layers.Dropout(dropout, name='audio_dropout'))
    cnn.add(layers.Dense(4, activation='softmax', name='audio_classification'))
    cnn.compile(optimizer=optimizers.Adam(learning_rate=learning_rate), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return cnn


def make_mlp(learning_rate, dropout):
    mlp = models.Sequential([
        layers.Input(shape=(75,)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(32, activation='relu'),
        layers.Dense(4, activation='softmax'),
    ], name='cnn_mlp_classifier')
    mlp.compile(optimizer=optimizers.Adam(learning_rate=learning_rate), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return mlp


def copy_audio_encoder(cnn, name='audio_encoder'):
    """Clone layers AND weights: never share layer instances between final models."""
    encoder = models.Sequential([layers.Input(shape=cnn.input_shape[1:])], name=name)
    for layer in cnn.layers:
        clone = layer.__class__.from_config(layer.get_config())
        encoder.add(clone)
        clone.set_weights(layer.get_weights())
        if layer.name == 'audio_features':
            break
    if encoder.output_shape[-1] != 64:
        raise ValueError('Missing 64-dimensional audio_features layer')
    encoder.trainable = False
    return encoder


def extract_audio_features(encoder, features, batch_size=8):
    return np.concatenate([np.asarray(encoder(np.asarray(features[i:i + batch_size]), training=False))
                           for i in range(0, len(features), batch_size)], axis=0)

## 8. 학습과 검증 손실 기준 후보 선택

각 후보를 학습 자료로 fit하고 `val_loss`로 조기 종료합니다. 최소 검증 손실이 나온 가중치를 복원하며, 동률이면 먼저 실행한 후보를 사용합니다.

In [ ]:
@contextmanager
def keras_scratch():
    with tempfile.TemporaryDirectory(dir=OUTPUT) as scratch:
        previous = tempfile.tempdir
        try:
            tempfile.tempdir = scratch
            yield
        finally:
            tempfile.tempdir = previous


def fit_model(name, factory, training_inputs, validation_inputs, directory):
    directory.mkdir(exist_ok=True)
    candidates = ([{'learning_rate': lr, 'dropout': dropout}
                   for lr in LEARNING_RATES for dropout in DROPOUT_RATES]
                  if AUTOTUNE else [DEFAULT_PARAMS])
    trials = []
    best_loss = np.inf
    for number, params in enumerate(candidates, start=1):
        if AUTOTUNE:
            tf.keras.utils.set_random_seed(SEED)
        model = factory(**params)
        print(f'{name} · trial {number}/{len(candidates)} · {params}')
        train = training_inputs()
        stop = callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True)
        fit_args = dict(validation_data=validation_inputs, epochs=MAX_EPOCHS,
                        callbacks=[stop], class_weight=class_weights, verbose=2)
        start = perf_counter()
        if isinstance(train, tuple):
            history = model.fit(*train, batch_size=BATCH_SIZE, shuffle=True, **fit_args)
        else:
            history = model.fit(train, shuffle=False, **fit_args)
        seconds = perf_counter() - start
        table = pd.DataFrame(history.history)
        table.insert(0, 'epoch', np.arange(1, len(table) + 1))
        if not np.isfinite(table.to_numpy()).all():
            raise ValueError(f'{name}: 학습 지표에 NaN 또는 무한대가 있습니다.')
        best = table.loc[table['val_loss'].idxmin()]
        trial = {'trial': number, **params, 'val_loss': float(best['val_loss']),
                 'best_epoch': int(best['epoch']), 'epochs': len(table),
                 'fit_seconds': seconds, 'epoch_seconds': seconds / len(table)}
        trials.append(trial)
        if trial['val_loss'] < best_loss:
            best_loss = trial['val_loss']
            chosen_model, chosen_history, chosen_trial = model, table, trial
    pd.DataFrame(trials).to_csv(directory / 'tuning_trials.csv', index=False)
    chosen_history.to_csv(directory / 'training_history.csv', index=False)
    save_json(directory / 'training_metrics.json', {'autotune': AUTOTUNE, **chosen_trial})
    return chosen_model, chosen_history, chosen_trial


def save_keras(model, path):
    with keras_scratch():
        temporary = Path(tempfile.tempdir) / path.name
        model.save(temporary)
        temporary.replace(path)

## 9. 음향 CNN 학습

음성만으로 네 클래스를 분류하는 CNN을 먼저 학습하고 저장합니다.

In [ ]:
cnn_dir = OUTPUT / 'cnn'
if saved_settings is None:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    cnn, cnn_history, cnn_trial = fit_model(
        'CNN', make_cnn, lambda: audio_dataset(prepared['train'], shuffle=True),
        audio_dataset(prepared['val']), cnn_dir,
    )
    save_keras(cnn, cnn_dir / 'model.keras')
else:
    cnn_history = pd.read_csv(cnn_dir / 'training_history.csv')

## 10. 고정 CNN 특징으로 MLP 학습

CNN에서 64차원 인코더를 복사해 고정하고, 학습·검증 WAV 특징에 환경값을 붙여 MLP를 따로 학습합니다.

In [ ]:
def make_mlp_inputs(part):
    audio_vectors = extract_audio_features(encoder, part['audio'], BATCH_SIZE)
    return np.concatenate([audio_vectors, part['environment']], axis=1)


model_dir = OUTPUT / 'cnn_mlp'
if saved_settings is None:
    encoder = copy_audio_encoder(cnn)
    mlp_inputs = {split: make_mlp_inputs(prepared[split]) for split in ('train', 'val')}
    mlp, mlp_history, mlp_trial = fit_model(
        'MLP', make_mlp, lambda: (mlp_inputs['train'], prepared['train']['labels']),
        (mlp_inputs['val'], prepared['val']['labels']), model_dir,
    )
else:
    with keras_scratch():
        encoder = models.load_model(model_dir / 'audio_encoder.keras', compile=False)
        mlp = models.load_model(model_dir / 'mlp.keras', compile=False)
    mlp_history = pd.read_csv(model_dir / 'training_history.csv')
    mlp_inputs = {}

### 모델 체크포인트와 추론 설정 저장

평가 전에 두 모델과 전처리 설정을 저장합니다. 이 셀을 마친 실행은 학습 없이 다시 평가할 수 있습니다.

In [ ]:
if saved_settings is None:
    # Invalidate completion and the bundle marker before replacing a checkpoint pair.
    (OUTPUT / 'completed.json').unlink(missing_ok=True)
    (model_dir / 'preprocessing.json').unlink(missing_ok=True)
    save_keras(encoder, model_dir / 'audio_encoder.keras')
    save_keras(mlp, model_dir / 'mlp.keras')
    # 기존 앱의 저장 형식에 필요한 이력 필드입니다. MLP 입력은 현재 환경값 11개입니다.
    app_environment = {
        **environment, 'version': 1, 'current_dim': 11, 'time_part_categories': list(range(6)),
        'sequence_length': 6, 'sequence_dim': 12, 'max_gap_hours': 2.0,
        'padding_value': 0.0, 'padding': 'right', 'delta_mean': 0.0, 'delta_std': 1.0,
    }
    settings = {
        'schema_version': 2, 'model_kind': 'cnn_mlp', 'architecture': 'GAP', 'experiment': 'B',
        'input_unit': 'whole_wav', 'audio': asdict(CONFIG), 'environment': app_environment,
        'quality': {'silence_rms': SILENCE_RMS, 'manual_exclusions': {}}, 'label_map': LABEL_MAP,
        'normalization': normalization, 'class_weights': class_weights, 'seed': SEED,
        'batch_size': BATCH_SIZE, 'autotune': AUTOTUNE,
        'monitoring': {'recording_seconds': 60, 'smoothing_window': 3, 'max_gap_seconds': 300}, 'cnn_trial': cnn_trial, 'mlp_trial': mlp_trial,
        'files': {'test_manifest': 'test_manifest.csv', 'test_environment': 'environment_test.csv'},
    }
    prepared['test']['rows'].sort_values(['recorded_at', 'file_path']).to_csv(
        model_dir / 'test_manifest.csv', index=False)
    observations[observations['split'].eq('test')].to_csv(model_dir / 'environment_test.csv', index=False)
    save_json(model_dir / 'preprocessing.json', settings)

## 11. 에포크별 학습 곡선

CNN과 MLP의 train·val accuracy와 loss를 표시하고 PNG로 저장합니다. 점선은 최소 검증 손실 에포크입니다.

In [ ]:
def plot_learning_curves(history_table, title):
    """Plot train and validation metrics over the epochs that actually ran."""
    from matplotlib.ticker import MaxNLocator, PercentFormatter

    if 'epoch' in history_table:
        epochs = history_table['epoch'].to_numpy()
    else:
        epochs = np.arange(1, len(history_table) + 1)
    fig, (loss_ax, accuracy_ax) = plt.subplots(
        1, 2, figsize=(10.5, 3.7), sharex=True, layout='constrained')
    fig.suptitle(title, fontweight='bold')
    for ax, metric, label in [(loss_ax, 'loss', 'Loss'), (accuracy_ax, 'accuracy', 'Accuracy')]:
        ax.plot(epochs, history_table[metric], 'o-', color='#22639D', label='Train', markersize=4)
        if f'val_{metric}' in history_table:
            ax.plot(epochs, history_table[f'val_{metric}'], 's--', color='#A34D21',
                    label='Validation', markersize=4)
        ax.set(xlabel='Epoch', title=label)
        ax.grid(axis='y', alpha=0.25)
        ax.set_axisbelow(True)
    loss_ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    if len(epochs) == 1:
        loss_ax.set_xticks(epochs)
    if 'val_loss' in history_table:
        best_epoch = epochs[history_table['val_loss'].to_numpy().argmin()]
        loss_ax.axvline(best_epoch, color='#69727D', linestyle=':',
                        label=f'Lowest val loss (epoch {best_epoch})')
    accuracy_ax.set_ylim(0, 1)
    accuracy_ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
    loss_ax.legend(loc='best')
    accuracy_ax.legend(loc='best')
    return fig


for history, name, directory in [(cnn_history, 'GAP CNN', cnn_dir), (mlp_history, 'CNN + MLP', model_dir)]:
    figure = plot_learning_curves(history, name)
    path = directory / 'learning_curves.png'
    figure.savefig(path, dpi=140, bbox_inches='tight')
    plt.close(figure)
    display(Image(filename=str(path)))

## 12. WAV·원본 단위 평가

선택된 MLP로 train·val·test를 평가합니다. 원본별로 클래스 확률을 평균한 뒤 한 번 판정합니다. 분할별 지표, 확률, 클래스 보고서와 혼동행렬을 저장합니다.

In [ ]:
def evaluate_split(split):
    wavs = prepared[split]['rows'].copy()
    scores = mlp.predict(mlp_inputs[split], batch_size=BATCH_SIZE, verbose=0)
    if scores.shape != (len(wavs), 4) or not np.isfinite(scores).all():
        raise ValueError(f'{split}: 예측 배열을 확인하세요.')
    wavs[SCORE_COLUMNS] = scores
    group = wavs.groupby('source_id', sort=True)
    sources = group[['label', 'hive_id', 'observed_at']].first().join(group[SCORE_COLUMNS].mean())
    sources['wav_count'] = group.size()
    directory = model_dir / split
    directory.mkdir(exist_ok=True)
    summaries, matrices = [], {}
    for unit, rows in [('wav', wavs.sort_values(['recorded_at', 'file_path'])),
                       ('source', sources.reset_index())]:
        rows = rows.assign(predicted_label=rows[SCORE_COLUMNS].to_numpy().argmax(axis=1))
        actual, predicted = rows['label'], rows['predicted_label']
        report = classification_report(actual, predicted, labels=CLASSES, output_dict=True, zero_division=0)
        matrix = confusion_matrix(actual, predicted, labels=CLASSES)
        metrics = {'split': split, 'unit': unit, 'accuracy': float(accuracy_score(actual, predicted)),
                   'macro_f1': report['macro avg']['f1-score'], 'support': len(rows),
                   **{f'recall_{i}': report[str(i)]['recall'] for i in CLASSES}}
        summaries.append(metrics)
        matrices[unit] = matrix
        rows.to_csv(directory / f'{unit}_predictions.csv', index=False)
        classes = pd.DataFrame({str(label): report[str(label)] for label in CLASSES}).T
        classes['support'] = classes['support'].astype(int)
        classes.to_csv(directory / f'{unit}_class_report.csv', index_label='class')
        pd.DataFrame(matrix, index=CLASSES, columns=CLASSES).to_csv(
            directory / f'{unit}_confusion_matrix.csv', index_label='true_label')
    save_json(directory / 'metrics.json', {row['unit']: row for row in summaries})
    return summaries, matrices


for split in SPLITS:
    if split not in mlp_inputs:
        mlp_inputs[split] = make_mlp_inputs(prepared[split])
evaluations = {split: evaluate_split(split) for split in SPLITS}
summary = pd.DataFrame([row for split in SPLITS for row in evaluations[split][0]])
summary.to_csv(OUTPUT / 'evaluation_summary.csv', index=False)
display(summary)

## 13. 테스트 결과 시각화

테스트의 WAV·원본 정확도와 macro F1, 혼동행렬을 표시하고 저장합니다.

In [ ]:
test_summary = summary[summary['split'].eq('test')].set_index('unit')
test_summary.to_csv(OUTPUT / 'test_summary.csv')
figure, axes = plt.subplots(1, 3, figsize=(11, 3.5), layout='constrained')
test_summary[['accuracy', 'macro_f1']].plot.bar(ax=axes[0], rot=0, ylim=(0, 1.08), color=['#22639D', '#A34D21'])
axes[0].set(title='Test accuracy and macro F1', xlabel='')
for bars in axes[0].containers:
    axes[0].bar_label(bars, fmt='%.3f', padding=2, fontsize=8)
for ax, unit in zip(axes[1:], ('wav', 'source')):
    ConfusionMatrixDisplay(evaluations['test'][1][unit], display_labels=CLASSES).plot(
        ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{unit.upper()} · n={int(test_summary.at[unit, "support"]):,}')
test_figure = OUTPUT / 'test_results.png'
figure.savefig(test_figure, dpi=145, bbox_inches='tight')
plt.close(figure)
display(Image(filename=str(test_figure)))

## 14. 저장 모델 재로딩 확인

저장 모델을 다시 불러와 같은 테스트 WAV의 예측이 일치하는지 확인합니다. Streamlit 앱에서는 모델 저장 루트에 실행 폴더를 지정하고 `cnn_mlp`를 선택합니다.

In [ ]:
with keras_scratch():
    saved_encoder = models.load_model(model_dir / 'audio_encoder.keras', compile=False)
    saved_mlp = models.load_model(model_dir / 'mlp.keras', compile=False)

first = prepared['test']
saved_audio = np.asarray(saved_encoder(first['audio'][:1], training=False))
saved_inputs = np.concatenate([saved_audio, first['environment'][:1]], axis=1)
expected = np.asarray(mlp(mlp_inputs['test'][:1], training=False))
restored = np.asarray(saved_mlp(saved_inputs, training=False))
np.testing.assert_allclose(restored, expected, rtol=1e-5, atol=2e-6)
save_json(OUTPUT / 'completed.json', {
    'experiment': 'B', 'architecture': 'GAP', 'autotune': AUTOTUNE,
    'reload_max_score_difference': float(np.max(np.abs(restored - expected))),
})
print(f'학습 곡선, 테스트 그림, 모델과 상세 결과  {OUTPUT}')